# Trend Visualizzazioni per Categoria Editoriale — DINAMICO
Analisi con date range personalizzabile per Taxi Drivers.  
Carica dati DAILY da Google Analytics 4 e permette di filtrarli per qualsiasi periodo.

## 1. Import Librerie

In [1]:
import os
import sys
from datetime import datetime, date, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from numpy.polynomial import Polynomial

# Widgets per interattività
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Path setup
_PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

# Import custom modules
from ga4_api.ga4_api import Ga4Client
from etl.page_and_screen_etl import PageAndScreenETLFactory

# Stile grafico uniforme
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})

print("✓ Librerie caricate")

ModuleNotFoundError: No module named 'ipywidgets'

## 2. Widget – Seleziona Date Range

In [ ]:
# Widget per selezionare il date range
start_date_input = widgets.DatePicker(
    value=date(2026, 4, 6),
    description='Start Date:',
    disabled=False
)

end_date_input = widgets.DatePicker(
    value=date(2026, 4, 12),
    description='End Date:',
    disabled=False
)

granularity_dropdown = widgets.Dropdown(
    options=['Daily', 'Weekly', 'Monthly'],
    value='Daily',
    description='Granularity:',
    disabled=False
)

load_button = widgets.Button(
    description='Load & Analyze',
    button_style='info',
    tooltip='Fetch data from GA4 and analyze',
    icon='refresh'
)

# Container
controls = widgets.VBox([
    widgets.HBox([start_date_input, end_date_input]),
    widgets.HBox([granularity_dropdown, load_button])
])

display(controls)

## 3. Caricamento Dati da GA4

In [ ]:
# Variabile globale per i dati
global_data = {'df': None, 'pivot': None, 'start_date': None, 'end_date': None}

def load_ga4_data(start_date: str, end_date: str) -> pd.DataFrame:
    """
    Carica dati da GA4 API per il date range specificato.
    Supporta granularità daily.
    """
    PROPERTY_ID = '394327334'
    DOMAIN = 'https://taxidrivers.it'
    
    print(f"🔄 Interrogazione GA4 ({start_date} → {end_date})...")
    
    try:
        ga4 = Ga4Client()
        df = ga4.run_query(
            property_id=PROPERTY_ID,
            dimensions=['pagePath', 'date'],  # Daily granularity
            metrics=['screenPageViews', 'activeUsers', 'engagedSessions', 'sessions', 'averageSessionDuration'],
            start_date=start_date,
            end_date=end_date
        )
        
        print(f"✓ Dati caricati: {len(df)} righe")
        
        # Pulisci con ETL
        print("🧹 Pulizia dati...")
        etl = PageAndScreenETLFactory.get_etl('en', df=df)
        etl.apply_transformations()
        df = etl.df
        print(f"✓ Dopo pulizia: {len(df)} righe")
        
        # Converti in datetime
        df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
        df = df.dropna(subset=['date_parsed'])
        
        # Mappa categoria da pagePath
        from map_ga4_categories import map_ga4_categories
        df['category'] = df['pagePath'].apply(map_ga4_categories)
        
        # Converti metriche a numeriche
        for col in ['screenPageViews', 'activeUsers', 'engagedSessions', 'sessions', 'averageSessionDuration']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        print("✓ Dati pronti per l'analisi")
        return df
        
    except Exception as e:
        print(f"❌ Errore: {e}")
        return None

print("✓ Funzione load_ga4_data definita")

## 4. Callback per il Button

In [ ]:
def on_load_button_click(b):
    """Callback per il button Load & Analyze"""
    output_area.clear_output(wait=True)
    
    with output_area:
        start_date = start_date_input.value
        end_date = end_date_input.value
        granularity = granularity_dropdown.value
        
        # Validazione
        if start_date > end_date:
            print("❌ Start Date deve essere prima di End Date")
            return
        
        # Carica dati
        df = load_ga4_data(
            start_date.strftime('%Y-%m-%d'),
            end_date.strftime('%Y-%m-%d')
        )
        
        if df is None or df.empty:
            print("❌ Nessun dato disponibile per il periodo specificato")
            return
        
        # Salva in global_data
        global_data['df'] = df
        global_data['start_date'] = start_date
        global_data['end_date'] = end_date
        global_data['granularity'] = granularity
        
        # Mostra info di base
        print(f"\n📊 Analisi del periodo: {start_date.strftime('%d/%m/%Y')} → {end_date.strftime('%d/%m/%Y')}")
        print(f"📈 Granularità: {granularity}")
        print(f"\n📋 Colonne disponibili:")
        print(df.columns.tolist())
        print(f"\n📌 Prime 5 righe:")
        display(df.head())
        print(f"\n✅ Dati caricati e pronti per l'analisi!")

# Collega il callback al button
load_button.on_click(on_load_button_click)

# Output area per mostrare i risultati
output_area = widgets.Output()
display(output_area)

print("✓ Callback definito")

## 5. Analisi: Views per Categoria

In [ ]:
analyze_button = widgets.Button(
    description='Analyze by Category',
    button_style='success',
    tooltip='Generate category analysis',
    icon='bar-chart'
)

def on_analyze_click(b):
    """Analizza i dati per categoria"""
    output_analysis.clear_output(wait=True)
    
    with output_analysis:
        if global_data['df'] is None or global_data['df'].empty:
            print("❌ Carica prima i dati con il pulsante 'Load & Analyze'")
            return
        
        df = global_data['df']
        granularity = global_data['granularity']
        
        # Aggrega per categoria e data (o per aggregazione temporale)
        if granularity == 'Daily':
            agg_df = df.groupby(['category', 'date_parsed']).agg({
                'screenPageViews': 'sum',
                'activeUsers': 'sum',
                'engagedSessions': 'sum',
                'sessions': 'sum',
                'averageSessionDuration': 'mean'
            }).reset_index()
        elif granularity == 'Weekly':
            df['week'] = df['date_parsed'].dt.to_period('W').apply(lambda x: x.start_time)
            agg_df = df.groupby(['category', 'week']).agg({
                'screenPageViews': 'sum',
                'activeUsers': 'sum',
                'engagedSessions': 'sum',
                'sessions': 'sum',
                'averageSessionDuration': 'mean'
            }).reset_index()
            agg_df.rename(columns={'week': 'date_parsed'}, inplace=True)
        else:  # Monthly
            df['month'] = df['date_parsed'].dt.to_period('M').apply(lambda x: x.start_time)
            agg_df = df.groupby(['category', 'month']).agg({
                'screenPageViews': 'sum',
                'activeUsers': 'sum',
                'engagedSessions': 'sum',
                'sessions': 'sum',
                'averageSessionDuration': 'mean'
            }).reset_index()
            agg_df.rename(columns={'month': 'date_parsed'}, inplace=True)
        
        # Calcola engagement rate
        agg_df['engagementRate'] = (agg_df['engagedSessions'] / agg_df['sessions'] * 100).round(2)
        
        # Pivot per visualizzazione
        pivot = agg_df.pivot_table(
            index='date_parsed',
            columns='category',
            values='screenPageViews',
            aggfunc='sum'
        )
        
        print(f"\n📊 Aggregazione {granularity}")
        print(f"Categorie presenti: {agg_df['category'].nunique()}")
        print(f"Periodi: {agg_df['date_parsed'].nunique()}")
        print(f"\n📈 Page Views per Categoria:")
        display(agg_df.groupby('category')['screenPageViews'].sum().sort_values(ascending=False))
        
        # Salva per plot
        global_data['agg_df'] = agg_df
        global_data['pivot'] = pivot
        
        # Plot: Views per categoria over time
        fig, ax = plt.subplots(figsize=(14, 6))
        for category in pivot.columns[:10]:  # Top 10 categorie
            ax.plot(pivot.index, pivot[category], marker='o', label=category, linewidth=2)
        ax.set_xlabel('Data')
        ax.set_ylabel('Page Views')
        ax.set_title(f'Page Views per Categoria ({global_data["granularity"]}) — {global_data["start_date"].strftime("%d/%m/%Y")} a {global_data["end_date"].strftime("%d/%m/%Y")}')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print("\n✅ Analisi completata!")

analyze_button.on_click(on_analyze_click)
output_analysis = widgets.Output()
display(analyze_button)
display(output_analysis)

print("✓ Analisi controls definiti")

## 6. Engagement Rate Analysis

In [ ]:
engagement_button = widgets.Button(
    description='Analyze Engagement',
    button_style='warning',
    tooltip='Engagement Rate analysis',
    icon='line-chart'
)

def on_engagement_click(b):
    """Analizza engagement rate"""
    output_engagement.clear_output(wait=True)
    
    with output_engagement:
        if global_data['agg_df'] is None:
            print("❌ Esegui prima 'Analyze by Category'")
            return
        
        agg_df = global_data['agg_df']
        
        # Engagement rate per categoria
        eng_pivot = agg_df.pivot_table(
            index='date_parsed',
            columns='category',
            values='engagementRate',
            aggfunc='mean'
        )
        
        print(f"\n📊 Engagement Rate per Categoria")
        print(agg_df.groupby('category')['engagementRate'].mean().sort_values(ascending=False))
        
        # Plot
        fig, ax = plt.subplots(figsize=(14, 6))
        for category in eng_pivot.columns[:10]:
            ax.plot(eng_pivot.index, eng_pivot[category], marker='s', label=category, linewidth=2)
        ax.set_xlabel('Data')
        ax.set_ylabel('Engagement Rate (%)')
        ax.set_title(f'Engagement Rate per Categoria — {global_data["start_date"].strftime("%d/%m/%Y")} a {global_data["end_date"].strftime("%d/%m/%Y")}')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print("\n✅ Engagement analysis completata!")

engagement_button.on_click(on_engagement_click)
output_engagement = widgets.Output()
display(engagement_button)
display(output_engagement)

print("✓ Engagement analysis controls definiti")

## 7. Top Articles per Categoria

In [ ]:
top_button = widgets.Button(
    description='Top Articles',
    button_style='danger',
    tooltip='Show top articles per category',
    icon='star'
)

def on_top_click(b):
    """Mostra top articoli per categoria"""
    output_top.clear_output(wait=True)
    
    with output_top:
        if global_data['df'] is None:
            print("❌ Carica prima i dati")
            return
        
        df = global_data['df']
        
        # Top 10 categorie
        top_categories = df.groupby('category')['screenPageViews'].sum().nlargest(10).index.tolist()
        
        for category in top_categories[:5]:  # Mostra solo le prime 5
            print(f"\n🎯 {category.upper()}")
            top_articles = df[df['category'] == category].groupby('pagePath').agg({
                'screenPageViews': 'sum',
                'activeUsers': 'sum',
                'averageSessionDuration': 'mean'
            }).sort_values('screenPageViews', ascending=False).head(5)
            print(top_articles)
            print()
        
        print("\n✅ Top articles analysis completata!")

top_button.on_click(on_top_click)
output_top = widgets.Output()
display(top_button)
display(output_top)

print("✓ Top articles controls definiti")